# Uniform-hybridisation approximation for the position PSDs (archived)

Tests an approximation in which every hybridisation function $R_{jk}(\omega)$ is replaced by
one common (mean) value $z$, so that the hybridised PSD of each dimension follows from the 1D
spectra in closed form. The first cell builds a 2×2 system ('equal spacing' Ω model, $g/2\pi =
-40$ kHz), plots the 1D and the full (LAM and numerical) position PSDs with their phonon
numbers and temperatures, then the ingredients $\mu_-$, $\eta_-$ and $\mu_-\eta_-|g|^2$ of the
first-order approximation. The second cell is an older draft of the same study (Δ/2π = −100 kHz,
an 8×2 system) that compares the approximated PSDs with the full ones.

**Runtime:** a few seconds with `FULL_SIZE = False` (the second cell on a 3×2 system).

**Updated Nov 2026** to current levisim:

- `np.trapz` → `np.trapezoid` (removed in NumPy 2.4);
- the full PSDs are now zero-point scaled (`zpf_scaled=True`), like the 1D spectra they are
  compared with and as `calculate_temperatures` expects; the original call used m²/Hz, which is
  why the stored output gave n = −0.5 phonons for every dimension of the full solution;
- the Im(μ−) curve plotted the real part twice; it now plots the imaginary part;
- second cell (never run in its stored form): `mtr = mtrap(...)` → `mtrap = MultiTrap(...)`,
  `Dimension.χM` → `Dimension.χ`, `calculate_all_hybridised_position_spectra(method,
  zpf_units=False)` → `all_position_power_spectra(method, zpf_scaled=True)`, and the 2024
  `'model': 'spread'` → `'Ω_model': 'equal spacing'` (the closest current model, same Ω_mid and
  Ω_span).

In [ ]:
# FULL_SIZE = True runs the second (older) draft on the 8×2 system it was written for (16
# dimensions, several minutes); False (the smoke-test setting) runs it on 3×2
FULL_SIZE = False

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gmean, hmean
from levisim import *

set_plt_rcParams_to_xmgrace_style()
# curves touch the left and right edges of the axes
plt.rcParams['axes.xmargin'] = 0

gas_kwargs = dict(P=1e-6*mbar, T=300*K, vth=500*metre/second)
cavity_kwargs = dict(Δ=-450*kHz, L=0.96*cm, Finesse=40_000, θhom=π/4)
toy_params = dict(Ω_model='equal spacing', toy_gx=-40*kHz, toy_gy=-40*kHz)
ω_range = np.arange(0, 500, 0.5) * kHz
system = '2×2'
Nd = system.split('×')
N, d = int(Nd[0]), int(Nd[1])
mtrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=d,
    ω_range=ω_range,
    gas_kwargs=gas_kwargs,
    cavity_kwargs=cavity_kwargs,
    toy_params=toy_params,
)

def Sth(dimension, ω):
    jn = dimension
    njn = jn.noise_phonons_approx

    return (
        np.abs(np.conjugate(jn.χ(-ω)))**2 * njn
        +
        np.abs(jn.χ(ω))**2 * (njn + 1)
    )


def Sjj_1D(mtrap, dimension, ω, zpf_units=False):
    jn = dimension
    κ, χC = mtrap.cavity.κ, mtrap.cavity.χC
    g = jn.g
    val = np.abs(jn.M(ω))**(-2) * (
        np.abs(jn.μm(ω))**2 * κ * g**2 * np.abs(χC(ω))**2
        +
        jn.Γ * Sth(jn, ω)
    )
    if zpf_units:
        return jn.zpf**2 * val
    return val


def calculate_temperatures(mtrap, plot_values, label):
    jn = mtrap.dimensions[label]
    ħΩ = ħ * jn.Ω

    SQQ_area = np.trapezoid(plot_values, mtrap.ω_range / Hz)
    njn = 0.5 * (SQQ_area - 1)
    print(f'{label=}')
    print(f'n{label} = {njn:.6f} phonons')
    T_via_phonons = njn * ħΩ / kB
    print(f'T{label} via phonons = {T_via_phonons:.6e} K')

    eqp_f = jn.m * jn.Ω**2 / kB
    q_variance = SQQ_area * jn.zpf**2
    T_via_variance = eqp_f * q_variance
    print(f'T{label} via variance = {T_via_variance:.6e} K')

    return {
        'phonons': T_via_phonons,
        'variance': T_via_variance,
    }


all_Rjk = []
for dim_j in list(mtrap.dimensions.values()):
    for dim_k in list(mtrap.dimensions.values()):

        if dim_j == dim_k:
            continue

        jn, km = dim_j.label, dim_k.label
        Rjk = HybridisationTerm(sign=1, pairs=[(jn, km)])
        print(term_string(Rjk))
        all_Rjk.append(Rjk)

assert len(all_Rjk) == (N * d)**2 - (N * d)

ylims = 1e-31, 1e-22
ylims = 1e-15, 1e3
# ylims = 1e-40, 1e-10
ALL_TEMPERATURES = {
    '1D': {},
    'full': {},
}

for label, dim in mtrap.dimensions.items():
    Sjj = [Sjj_1D(mtrap, dim, ω) for ω in mtrap.ω_range]
    plt.plot(mtrap.ω_range / kHz, Sjj, label=label)
    ALL_TEMPERATURES['1D'][dim.label] = calculate_temperatures(mtrap, Sjj, dim.label)
    # plt.vlines(dim.Ω / kHz, 0, 1e-27, color='k', lw=0.5, ls='--')
plt.title('1D spectra')
plt.yscale('log')
plt.ylim(*ylims)
plt.legend()
plt.show()

for method in [
    QuantumLinearTheoryMethod.LINEAR_AMPLIFIER_MODEL,
    QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION,
]:
    # zero-point scaled, like Sjj_1D above and as calculate_temperatures expects (the original
    # used zpf_scaled=False, i.e. m²/Hz, which gave n = -0.5 phonons for every dimension)
    all_full_Sjj, _ = mtrap.all_position_power_spectra(method, zpf_scaled=True)
    plt.plot(mtrap.ω_range / kHz, all_full_Sjj, label=mtrap.labels)
    for ii, label in enumerate(mtrap.labels):
        ALL_TEMPERATURES['full'][label] = calculate_temperatures(mtrap, all_full_Sjj[:, ii], label)
    plt.title(f'full solution - {method}')
    plt.yscale('log')
    plt.ylim(*ylims)
    plt.grid()
    plt.legend()
    plt.show()

# all_temperatures = {}
# LAM_temps = compare_methods(mtrap, [QuantumLinearTheoryMethod.LINEAR_AMPLIFIER_MODEL], ['PSDs'], return_PSD_temperatures=True)
# all_temperatures['LAM'] = LAM_temps

# methods = [QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION]
# orders = list(range(1, NN + 1))[::-1]
# for order in orders:
#     mtrap.truncation_order = order
#     print('#######################################################################################')
#     print(f'{mtrap.truncation_order=}')
#     print(f'{len(mtrap.deranged_determinant_terms)=}')
#     print(f'{len(mtrap.deranged_adjugate_terms["x1x1"])=}')
#     print(f'{len(mtrap.deranged_adjugate_terms["x1x2"])=}')
#     temps = compare_methods(mtrap, [QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION], ['PSDs'], return_PSD_temperatures=True)
#     print(temps)
#     all_temperatures[order] = temps

def mean_R(all_Rjk, ω, method):
    all_Rjk_values = [mtrap.hybridisation_term_value(Rjk, ω) for Rjk in all_Rjk]
    if method == 'arithmetic':
        return np.mean(all_Rjk_values)
    if method == 'geometric':
        return gmean(all_Rjk_values)
    # NOTE: hmean doesn't accept negative values...
    if method == 'harmonic':
        return hmean(all_Rjk_values)
    if method == 'max abs':
        return np.abs(all_Rjk_values).max()
    if method == 'argmax abs':
        return all_Rjk_values[np.abs(all_Rjk_values).argmax()]

def full_approx_uniform_Sjj(mtrap, dimension, ω):
    N = mtrap.num_dimensions
    dim = dimension
    # approximate this with its first order taylor series
    # z = R ~ -μ.η.|g|² + ...
    z = -dim.μm(ω) * mtrap.cavity.ηm(ω) * np.abs(dim.G)**2
    # approximate this with its first order taylor series, too
    # 1/(1 - (N-1)z) ~ 1 + (N-1)z + ...
    return np.abs((1 + (N - 1) * z))**2 * Sjj_1D(mtrap, dimension, ω)


label = 'x1'
dim = mtrap.dimensions[label]

μm_vals = np.array([dim.μm(ω) for ω in mtrap.ω_range])
plt.plot(mtrap.ω_range / kHz, np.real(μm_vals), label='Re(μ-)')
plt.plot(mtrap.ω_range / kHz, np.imag(μm_vals), label='Im(μ-)')
plt.plot(mtrap.ω_range / kHz, np.abs(μm_vals), label='abs(μ-)')
plt.yscale('log')
plt.legend()
plt.grid()
plt.show()

ηm_vals = np.array([mtrap.cavity.ηm(ω) for ω in mtrap.ω_range])
plt.plot(mtrap.ω_range / kHz, np.real(ηm_vals), label='Re(η-)')
plt.plot(mtrap.ω_range / kHz, np.imag(ηm_vals), label='Im(η-)')
plt.plot(mtrap.ω_range / kHz, np.abs(ηm_vals), label='abs(η-)')
# plt.yscale('log')
plt.legend()
plt.grid()
plt.show()


plt.plot(mtrap.ω_range / kHz, np.real(μm_vals * ηm_vals), label='Re(μ-.η-)')
plt.plot(mtrap.ω_range / kHz, np.imag(μm_vals * ηm_vals), label='Im(μ-.η-)')
plt.plot(mtrap.ω_range / kHz, np.abs(μm_vals * ηm_vals), label='abs(μ-.η-)')
plt.yscale('log')
plt.legend()
plt.grid()
plt.show()

# |g|²
g2 = np.abs(dim.G)**2
plt.plot(mtrap.ω_range / kHz, np.real(μm_vals * ηm_vals * g2), label='Re(μ-.η-.|g|²)')
plt.plot(mtrap.ω_range / kHz, np.imag(μm_vals * ηm_vals * g2), label='Im(μ-.η-.|g|²)')
plt.plot(mtrap.ω_range / kHz, np.abs(μm_vals * ηm_vals * g2), label='abs(μ-.η-.|g|²)')
plt.yscale('log')
plt.legend()
plt.grid()
plt.show()

def x_val(mtrap, dimension, ω):
    return dimension.μm(ω) * mtrap.cavity.ηm(ω) * np.abs(dimension.G)**2

def z_val(mtrap, dimension, ω):
    x = x_val(mtrap, dimension, ω)
    return -x / (1 + x)


x_vals = [x_val(mtrap, dim, ω) for ω in mtrap.ω_range]
z_vals = [z_val(mtrap, dim, ω) for ω in mtrap.ω_range]

# plt.plot(mtrap.ω_range / kHz, np.real(x_vals), label='Re(x)')
# plt.plot(mtrap.ω_range / kHz, np.imag(x_vals), label='Im(x)')
plt.plot(mtrap.ω_range / kHz, np.abs(x_vals), label='abs(x)')
plt.yscale('log')
plt.legend()
plt.show()

plt.plot(mtrap.ω_range / kHz, np.real(z_vals), label='Re(z)')
plt.plot(mtrap.ω_range / kHz, np.imag(z_vals), label='Im(z)')
plt.plot(mtrap.ω_range / kHz, np.abs(z_vals), label='abs(z)')
plt.legend()
plt.show()

max_abs_x = np.max(x_vals)
max_abs_z = np.max(z_vals)
print(f'{max_abs_x=}')
print(f'{max_abs_z=}')

Sjj = [full_approx_uniform_Sjj(mtrap, dim, ω) for ω in mtrap.ω_range]
plt.plot(mtrap.ω_range / kHz, Sjj, label=label)
ALL_TEMPERATURES['full approx uniform'] = {}
ALL_TEMPERATURES['full approx uniform'][label] = calculate_temperatures(mtrap, Sjj, label)
plt.title(f'full approx uniform')
plt.yscale('log')
plt.ylim(*ylims)
plt.legend()
plt.grid()
plt.show()

def exact_uniform_Sjj(mtrap, dimension, ω):
    N = mtrap.num_dimensions
    dim = dimension
    x = dim.μm(ω) * mtrap.cavity.ηm(ω) * np.abs(dim.G)**2
    z = -x / (1 + x)
    return np.abs((1 - (N - 1) * z))**(-2) * Sjj_1D(mtrap, dimension, ω)

label = 'x1'
dim = mtrap.dimensions[label]
Sjj = [exact_uniform_Sjj(mtrap, dim, ω) for ω in mtrap.ω_range]
plt.plot(mtrap.ω_range / kHz, Sjj, label=label)
ALL_TEMPERATURES['exact uniform'] = {}
ALL_TEMPERATURES['exact uniform'][label] = calculate_temperatures(mtrap, Sjj, label)
plt.title(f'exact uniform')
plt.yscale('log')
plt.ylim(*ylims)
plt.legend()
plt.grid()
plt.show()

def uniform_Sjj(mtrap, dimension, ω, mean_method, overall_method):
    N = mtrap.num_dimensions

    all_other_dims = [dim for dim in mtrap.dimensions.values()]
    all_other_dims.remove(dimension)

    z = mean_R(all_Rjk, ω, method=mean_method)

    if overall_method == 1:
        return np.abs(
            (1 - (N - 2) * z )
            /
            (1 - (N - 2) * z - (N - 1) * z**2)
        )**2 * Sjj_1D(mtrap, dimension, ω)
        + np.abs(
            (z)
            /
            (1 - (N - 2) * z - (N - 1) * z**2)
        )**2 * np.sum(
            [Sjj_1D(mtrap, other_dim, ω) for other_dim in all_other_dims]
        )

    if overall_method == 2:
        part_1 = np.abs(
            (1 - (N - 2) * z )
            /
            (1 - (N - 2) * z - (N - 1) * z**2)
        )**2 * Sjj_1D(mtrap, dimension, ω)

        part_2 = 0 + 0j
        for other_dim in all_other_dims:
            for Rjk in all_Rjk:
                jn, km = Rjk.pairs[0]
                if Rjk.pairs[0] == (dimension.label, other_dim.label):
                    break
            # if ω == 0:
            #     print(Rjk)
            Rjk_value = mtrap.hybridisation_term_value(Rjk, ω)
            part_2 += (
                np.abs(
                    (Rjk_value)
                    /
                    (1 - (N - 2) * Rjk_value - (N - 1) * Rjk_value**2)
                )**2 * Sjj_1D(mtrap, other_dim, ω)
            )

        return part_1 + part_2

for overall_method in [
    1,
    # 2,
]:
    for mean_method in [
        'arithmetic',
        # 'geometric',
        # 'max abs',
        # 'argmax abs',
    ]:
        ALL_TEMPERATURES[f'{overall_method} - {mean_method}'] = {}
        for label, dim in mtrap.dimensions.items():
            Sjj = [uniform_Sjj(mtrap, dim, ω, mean_method=mean_method, overall_method=overall_method) for ω in mtrap.ω_range]
            plt.plot(mtrap.ω_range / kHz, Sjj, label=label)
            ALL_TEMPERATURES[f'{overall_method} - {mean_method}'][label] = calculate_temperatures(mtrap, Sjj, label)
            # plt.vlines(dim.Ω / kHz, 0, 1e-27, color='k', lw=0.5, ls='--')
        plt.title(f'uniform ' + r'$z=\bar{R}$' + f' ({overall_method}: {mean_method})')
        plt.yscale('log')
        plt.ylim(*ylims)
        plt.legend()
        plt.grid()
        plt.show()

from pprint import pprint
pprint(ALL_TEMPERATURES)

## Older draft: the approximation compared with the full PSDs

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gmean, hmean

from levisim import *
from levisim.multiphysics.hybridisation import term_string



toy_params = {
    # the 2024 'spread' model, whose closest current form is 'equal spacing' (x and y frequencies
    # evenly filling Ω_mid ± Ω_span/2; 'spread' set Ωy = Ωx - Ω_gap but no Ω_gap was given here)
    'Ω_model': 'equal spacing',
    'Ω_mid': 300*kHz,
    'Ω_span': 100*kHz,

    # 'model': 'pairs',
    # 'first_Ωx': 312*kHz,
    # 'Ω_gap': 15*kHz,
    # 'Ωz_to_Ωx_multiplier': 10,

    # 'model': 'equal',
    # 'equal_Ω': 300*kHz,

    # 'model': 'custom',
    # 'Ωxs': np.array([300, 302]) * kHz,
    # 'Ωys': np.array([301, 303]) * kHz,

    'toy_gx': -40*kHz,
    'toy_gy': -40*kHz,

    'use_Γ_formula': True,
}
cavity_kwargs = {
    'Δ': -100*kHz,
    'L': 0.96*cm,
    'Finesse': 40_000,
    'θhom': π/4,
}
gas_kwargs = {
    'P': 1e-6*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}
N, ax = 2, 3
N, ax = 3, 2
N, ax = 3, 2
N, ax = 8, 2
# a 16-dimensional system takes minutes; FULL_SIZE = False uses the 3×2 option above instead
if not FULL_SIZE:
    N, ax = 3, 2
NN = N * ax
ω_low, ω_high, ω_step = 0, 800, 0.5
mtrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=ax,
    ω_range=np.arange(ω_low, ω_high, ω_step) * kHz,

    cavity_kwargs=cavity_kwargs,
    gas_kwargs=gas_kwargs,
    toy_params=toy_params,

    load_in_power_series=False,
)
mtrap.info
mtrap.info_per_trap
mtrap.info_per_dimension


def Sth(dimension, ω):
    jn = dimension
    njn = jn.noise_phonons_approx

    return (
        np.abs(np.conjugate(jn.χ(-ω)))**2 * njn
        +
        np.abs(jn.χ(ω))**2 * (njn + 1)
    )


def Sjj_1D(mtrap, dimension, ω, zpf_units=False):
    jn = dimension
    κ, χC = mtrap.cavity.κ, mtrap.cavity.χC
    g = jn.g
    val = np.abs(jn.M(ω))**(-2) * (
        np.abs(jn.μm(ω))**2 * κ * g**2 * np.abs(χC(ω))**2
        +
        jn.Γ * Sth(jn, ω)
    )
    if zpf_units:
        return jn.zpf**2 * val
    return val


def calculate_temperatures(mtrap, plot_values, label):
    jn = mtrap.dimensions[label]
    ħΩ = ħ * jn.Ω

    SQQ_area = np.trapezoid(plot_values, mtrap.ω_range / Hz)
    njn = 0.5 * (SQQ_area - 1)
    print(f'{label=}')
    print(f'n{label} = {njn:.6f} phonons')
    T_via_phonons = njn * ħΩ / kB
    print(f'T{label} via phonons = {T_via_phonons:.6e} K')

    eqp_f = jn.m * jn.Ω**2 / kB
    q_variance = SQQ_area * jn.zpf**2
    T_via_variance = eqp_f * q_variance
    print(f'T{label} via variance = {T_via_variance:.6e} K')

    return {
        'phonons': T_via_phonons,
        'variance': T_via_variance,
    }


all_Rjk = []
for dim_j in list(mtrap.dimensions.values()):
    for dim_k in list(mtrap.dimensions.values()):

        if dim_j == dim_k:
            continue

        jn, km = dim_j.label, dim_k.label
        Rjk = HybridisationTerm(sign=1, pairs=[(jn, km)])
        print(term_string(Rjk))
        all_Rjk.append(Rjk)

assert len(all_Rjk) == NN**2 - NN


ylims = 1e-31, 1e-22
ylims = 1e-9, 1e0
# ylims = 1e-40, 1e-10
ALL_TEMPERATURES = {
    '1D': {},
    'full': {},
}

for label, dim in mtrap.dimensions.items():
    Sjj = [Sjj_1D(mtrap, dim, ω) for ω in mtrap.ω_range]
    plt.plot(mtrap.ω_range / kHz, Sjj, label=label)
    ALL_TEMPERATURES['1D'][dim.label] = calculate_temperatures(mtrap, Sjj, dim.label)
    # plt.vlines(dim.Ω / kHz, 0, 1e-27, color='k', lw=0.5, ls='--')
plt.title('1D spectra')
plt.yscale('log')
plt.ylim(*ylims)
plt.legend()
plt.show()

for method in [
    QuantumLinearTheoryMethod.LINEAR_AMPLIFIER_MODEL,
    QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION,
]:
    # zpf_units=False meant zero-point scaled, i.e. zpf_scaled=True today
    all_full_Sjj, _ = mtrap.all_position_power_spectra(method, zpf_scaled=True)
    plt.plot(mtrap.ω_range / kHz, all_full_Sjj, label=mtrap.labels)
    for ii, label in enumerate(mtrap.labels):
        ALL_TEMPERATURES['full'][label] = calculate_temperatures(mtrap, all_full_Sjj[:, ii], label)
    plt.title(f'full solution - {method}')
    plt.yscale('log')
    plt.ylim(*ylims)
    plt.grid()
    plt.legend()
    plt.show()

# all_temperatures = {}
# LAM_temps = compare_methods(mtrap, [QuantumLinearTheoryMethod.LINEAR_AMPLIFIER_MODEL], ['PSDs'], return_PSD_temperatures=True)
# all_temperatures['LAM'] = LAM_temps

# methods = [QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION]
# orders = list(range(1, NN + 1))[::-1]
# for order in orders:
#     mtrap.truncation_order = order
#     print('#######################################################################################')
#     print(f'{mtrap.truncation_order=}')
#     print(f'{len(mtrap.deranged_determinant_terms)=}')
#     print(f'{len(mtrap.deranged_adjugate_terms["x1x1"])=}')
#     print(f'{len(mtrap.deranged_adjugate_terms["x1x2"])=}')
#     temps = compare_methods(mtrap, [QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION], ['PSDs'], return_PSD_temperatures=True)
#     print(temps)
#     all_temperatures[order] = temps

def mean_R(all_Rjk, ω, method):
    all_Rjk_values = [mtrap.hybridisation_term_value(Rjk, ω) for Rjk in all_Rjk]
    if method == 'arithmetic':
        return np.mean(all_Rjk_values)
    if method == 'geometric':
        return gmean(all_Rjk_values)
    # NOTE: hmean doesn't accept negative values...
    if method == 'harmonic':
        return hmean(all_Rjk_values)
    if method == 'max abs':
        return np.abs(all_Rjk_values).max()
    if method == 'argmax abs':
        return all_Rjk_values[np.abs(all_Rjk_values).argmax()]


def uniform_Sjj(mtrap, dimension, ω, mean_method, overall_method):
    N = mtrap.num_dimensions

    all_other_dims = [dim for dim in mtrap.dimensions.values()]
    all_other_dims.remove(dimension)

    z = mean_R(all_Rjk, ω, method=mean_method)

    if overall_method == 1:
        return np.abs(
            (1 - (N - 2) * z )
            /
            (1 - (N - 2) * z - (N - 1) * z**2)
        )**2 * Sjj_1D(mtrap, dimension, ω)
        + np.abs(
            (z)
            /
            (1 - (N - 2) * z - (N - 1) * z**2)
        )**2 * np.sum(
            [Sjj_1D(mtrap, other_dim, ω) for other_dim in all_other_dims]
        )

    if overall_method == 2:
        part_1 = np.abs(
            (1 - (N - 2) * z )
            /
            (1 - (N - 2) * z - (N - 1) * z**2)
        )**2 * Sjj_1D(mtrap, dimension, ω)

        part_2 = 0 + 0j
        for other_dim in all_other_dims:
            for Rjk in all_Rjk:
                jn, km = Rjk.pairs[0]
                if Rjk.pairs[0] == (dimension.label, other_dim.label):
                    break
            # if ω == 0:
            #     print(Rjk)
            Rjk_value = mtrap.hybridisation_term_value(Rjk, ω)
            part_2 += (
                np.abs(
                    (Rjk_value)
                    /
                    (1 - (N - 2) * Rjk_value - (N - 1) * Rjk_value**2)
                )**2 * Sjj_1D(mtrap, other_dim, ω)
            )

        return part_1 + part_2

for overall_method in [
    1,
    # 2,
]:
    for mean_method in [
        'arithmetic',
        # 'geometric',
        # 'max abs',
        # 'argmax abs',
    ]:
        ALL_TEMPERATURES[f'{overall_method} - {mean_method}'] = {}
        for label, dim in mtrap.dimensions.items():
            Sjj = [uniform_Sjj(mtrap, dim, ω, mean_method=mean_method, overall_method=overall_method) for ω in mtrap.ω_range]
            plt.plot(mtrap.ω_range / kHz, Sjj, label=label)
            ALL_TEMPERATURES[f'{overall_method} - {mean_method}'][label] = calculate_temperatures(mtrap, Sjj, label)
            # plt.vlines(dim.Ω / kHz, 0, 1e-27, color='k', lw=0.5, ls='--')
        plt.title(f'uniform ' + r'$z=\bar{R}$' + f' ({overall_method}: {mean_method})')
        plt.yscale('log')
        plt.ylim(*ylims)
        plt.legend()
        plt.grid()
        plt.show()

from pprint import pprint
pprint(ALL_TEMPERATURES)